# ntl-stagecheck-graft

The GLM stage of `ntl-graft-gbm` (head `gbmpair`, feiyu26's graft) run end to end on 120 train passages; its predictions must match the offline ones of exp050.

In [ ]:
import glob, os, time
import numpy as np, pandas as pd
os.makedirs('/kaggle/temp', exist_ok=True)
TRAIN = next(p for p in glob.glob('/kaggle/input/**/train.csv', recursive=True) if 'name-that-llm' in p)
DATA = os.path.dirname(TRAIN)   # the V1 cells smoke-test the first three rows of DATA/test.csv


In [ ]:
# ---- ours, V1: likelihood under GLM-5.3-Flash and GLM-5.3, after the 0.684 S10 (glm689's stage) ----
from pathlib import Path
V1_ENABLED    = True
V1_DEADLINE_H = 8.8    # all V1 work ends this many hours after the notebook started (the limit is 9 h)
V1_PACK, V1_UBATCH_DIV, V1_SEQ_BATCH, V1_PREFETCH = False, 1, 8, False
# measured here, not glm689's: 360 texts on the scale run cost flash 979 s and glm53 1,926 s for 47,098 token
# rows, i.e. ~0.021 and ~0.041 s per token row and no meaningful fixed cost; the published constants are half
# that, and an optimistic estimate is what makes the stage over-cover and lose the whole run to its budget
V1_COST = dict(fixed_s_flash=100.0, row_s_flash=0.0225, fixed_s_glm53=100.0, row_s_glm53=0.0442)
V1_RERUN = os.getenv('KAGGLE_IS_COMPETITION_RERUN') is not None
V1_TEST_CSV = os.path.join(DATA, 'test.csv')


In [ ]:
# ---- ours, V1 setup and checks (docs/12_v1_verosimilitud_glm.md, fase 2) ----
import base64, glob, hashlib, io, json, os, shutil, subprocess, sys, tarfile, time
V1_PAYLOAD = "..."
_raw = base64.b64decode(V1_PAYLOAD)
assert hashlib.sha256(_raw).hexdigest() == "bfe4fc235765f76ee43ecea92bd27523949f25bc335d58ceabbfb266e3339985"
V1_DIR = Path('/kaggle/temp/v1'); V1_DIR.mkdir(parents=True, exist_ok=True)
tarfile.open(fileobj=io.BytesIO(_raw), mode='r:gz').extractall(V1_DIR)
V1_OUT = Path('/kaggle/working/v1'); V1_OUT.mkdir(parents=True, exist_ok=True)

def _sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()

# binaries: the ntl-bin/ folder of the attached ntl-glm-likelihood-train output, exactly this build
# plus its CUDA 13 runtime (ntl-bin/lib): this image (S10's, pinned) only has the CUDA 12.8 toolkit
_cands = sorted(glob.glob('/kaggle/input/**/ntl-bin/ntl-segrun', recursive=True))
_ok = [p for p in _cands if _sha(p) == "9c39e3afc77c64c1930f438fef50f297a6a8a649252ab4af9f618804be84cc8c" and Path(p).with_name('lib').joinpath('libcublasLt.so.13').exists()]
assert _ok, f'V1: no ntl-segrun with the expected sha256 and a lib/ folder among {_cands}'
V1_BIN = Path('/kaggle/temp/v1-bin'); V1_BIN.mkdir(exist_ok=True)
shutil.copy(_ok[0], V1_BIN / 'ntl-segrun'); os.chmod(V1_BIN / 'ntl-segrun', 0o755)
V1_LIB = Path(_ok[0]).with_name('lib')
# only for the V1 processes (S10's torch keeps its own CUDA 12.8 libraries); LD_LIBRARY_PATH wins over the RUNPATH
V1_ENV = dict(os.environ, LD_LIBRARY_PATH=str(V1_LIB) + ':' + os.environ.get('LD_LIBRARY_PATH', ''))

# segments of the two models (public layer-segment GGUFs), one folder of links per model
def _segs(pattern, n, name):
    seen = {}
    for p in sorted(glob.glob('/kaggle/input/**/' + pattern, recursive=True)):
        seen.setdefault(os.path.basename(p), p)
    assert len(seen) == n, f'V1: {len(seen)} {name} segments, expected {n}'
    d = Path(f'/kaggle/temp/v1-segs-{name}'); d.mkdir(exist_ok=True)
    for b, p in seen.items():
        if not (d / b).exists(): (d / b).symlink_to(p)
    return d
V1_FLASH = _segs('flash_iq3xxs-seg*-of-06-*.gguf', 6, 'flash')
V1_GLM53 = _segs('glm53_iq3xxs-seg*-of-19-*.gguf', 19, 'glm53')

# checks before S10 runs, so a broken V1 fails in minutes: tokenize always (loads the CUDA libraries),
# and in a commit or a rehearsal a GPU smoke test (Flash segment 0 on 3 rows: model load, one decode)
if V1_ENABLED:
    sys.path.insert(0, str(V1_DIR)); import segio
    _t = pd.read_csv(V1_TEST_CSV, dtype={'id': str}).head(3)
    segio.write_texts('/kaggle/temp/v1_smoke_texts.bin', _t.text.fillna('').astype(str).tolist())
    _seg0 = sorted(V1_FLASH.glob('*.gguf'))[0]
    _r = subprocess.run(f'{V1_BIN}/ntl-segrun tokenize --model {_seg0} --texts /kaggle/temp/v1_smoke_texts.bin --out /kaggle/temp/v1_smoke_tok.bin '
                        f'--max-text-tokens 160 --prefix "[gMASK]<sop><|system|>Reasoning Effort: Max<|user|><|assistant|><think></think>"' +
                        ('' if V1_RERUN else f' && {V1_BIN}/ntl-segrun run --model {_seg0} --tokens /kaggle/temp/v1_smoke_tok.bin --out /kaggle/temp/v1_smoke_state.bin '
                        f'--state-dtype f32 --seq-batch 8 --split-mid'), shell=True, capture_output=True, text=True, env=V1_ENV)
    (V1_OUT / 'smoke.log').write_text(_r.stdout + _r.stderr)
    print('\n'.join(l for l in (_r.stdout + _r.stderr).splitlines() if l.startswith('ntl-segrun')), flush=True)
    assert _r.returncode == 0 and (V1_RERUN or 'state max |x|' in _r.stderr), 'V1 smoke test failed, see /kaggle/working/v1/smoke.log'
    print('V1 checks OK: binary sha256 + CUDA libraries, 6 + 19 segments,', 'tokenizer' if V1_RERUN else 'GPU smoke test', flush=True)

In [ ]:
import json, subprocess, sys, time
import numpy as np, pandas as pd
from pathlib import Path
CHECK = json.loads('{"ids": ["032c15c0abec", "050bfca264cc", "0663836c7892", "094705a424ee", "0959253d546b", "0994248d255d", "10936e416e17", "11770123c25d", "145abfe0654d", "18aac2816e9e", "19b5f954f562", "1f317e09e595", "1f39fd737ff2", "26cbebe69d9e", "2b4c16598e0a", "2baccb5c1c88", "2dc574b09e59", "3164365b3bfc", "325fa3fd97dc", "34082b6a2ff0", "361cc72ec05b", "365f33009e27", "36c15019e1df", "3abc8b8fa3b3", "3b2a05a616fd", "3c788a70c26f", "3e70864eb8c4", "415a5c18d9f1", "46da3aa3c840", "471e941fc5a8", "478e27479e0e", "4a2caf078aa3", "4c4235c98f21", "4e2c924764ae", "506cc037e03d", "5445f9438edf", "5ad5a0214b61", "5ae5e0352d46", "61714942b7a5", "61f329ac925b", "6545c9754511", "66cc4fbc709f", "684d6e83ff67", "68a802b59fef", "6b4d42ec2230", "6c6767295f85", "6d826f0a6c54", "6e8b5e4b3d68", "6fc51978f9e5", "6fee3079acf3", "702f4268f2a5", "7922de5c29ca", "79cc731d5512", "7b48dd04ba7e", "8280378024f1", "8622d9252d72", "8950e47cb1fc", "89d8382c8c70", "8cd9b1d040a0", "92295a4896d8", "92594e9c69a4", "950981db298a", "95441f16ba27", "9650b548d3c3", "9bacae2e38c8", "9e58eb11a830", "a31835ac1877", "a4084e6740f6", "a5d46f1dcdc5", "ab10e3df96ac", "b271a81ec0a5", "bcc5944f94fe", "bd9686531247", "bdbcf2f48026", "c1478e6e7518", "c4a88dc254f4", "c7c1ff6c0407", "ca23d509d6cb", "cacc4db59f48", "ccbeb5cd7e08", "ce0083c48dec", "d413f9b9b139", "d4be4d33d4a3", "d52bc7b9ac5d", "d5665b99d56d", "d9a234ef935f", "db3f4a04db54", "db874dae3056", "e0082c9d52e0", "e04e402d511c", "e234eecef3b0", "e3546e666991", "e36f3339ebda", "e49571b21cec", "e5861a399d53", "e599c0b5f5f1", "e61e8ea9f601", "e63bcbb6e37c", "e6834c3cd6db", "e6ed34a27e93", "e9ff3857c324", "ead512ab8605", "eb9ebd034df9", "ebf22557e0ee", "ed8aa46c6c52", "eebcfb7b8814", "efe447f6221d", "f1c5b7abc0d2", "f21e4a81ecbf", "f3374b94f1b6", "f55ac93f7f1c", "f5f8c20330bf", "f76e637a8a8b", "f8fe087cdee6", "f97daadafe5d", "f9aa3bf0cfd2", "fb07ed3f0ede", "fcf958ea694f", "fd3b972557d5", "fde38289c3ce"], "labels": ["claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5", "gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash", "glm-5.3", "glm-5.3-flash", "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra", "kimi-k3", "qwen3.8-max"], "logits": [[-2.119807, -6.52863, -4.610564, -8.617945, -4.02983, -8.52956, -8.216423, -8.401681, -8.850446, -2.100167, -0.612232, -5.890839, -6.577459, -7.00143, -7.244305, -7.463541, -1.861498, -3.756099], [-1.77232, -4.633889, -3.584447, -3.621565, -4.052189, -5.632128, -4.357251, -5.475311, -4.381515, -1.421078, -1.261131, -5.731853, -5.358811, -4.545807, -5.512622, -5.317212, -1.958793, -3.813938], [-1.947313, -3.410861, -2.44354, -3.280058, -2.56932, -5.702316, -5.454545, -6.126666, -4.950128, -1.420313, -1.249436, -5.743213, -5.849784, -6.737622, -6.554162, -6.998157, -3.257749, -3.506737], [-1.819574, -1.990083, -2.506609, -5.371565, -4.131112, -6.738722, -5.404818, -6.092889, -6.446073, -1.162619, -1.446597, -6.753192, -5.875023, -6.8013, -6.560647, -6.567679, -3.520749, -5.541976], [-3.844183, -4.85174, -3.960103, -6.536958, -5.460129, -7.439499, -5.452126, -5.961216, -6.326155, -0.835645, -1.168176, -7.186108, -4.465206, -6.086605, -6.86456, -7.213934, -2.463746, -2.398864], [-2.147084, -5.483208, -6.003618, -5.150897, -4.412168, -6.988578, -6.58907, -6.224215, -6.818188, -1.47926, -0.825907, -5.052771, -3.926044, -4.221324, -5.434808, -4.856761, -2.044375, -5.255323], [-1.955077, -6.828154, -4.396556, -5.028756, -4.158898, -8.19726, -7.529067, -7.039419, -8.066607, -0.918636, -0.979877, -7.077829, -6.707598, -6.779509, -7.498315, -7.415111, -3.19746, -6.705006], [-3.767483, -4.98962, -4.20268, -2.061255, -4.153581, -7.476127, -7.023011, -6.099771, -6.592616, -1.219761, -0.826727, -7.512349, -6.775089, -7.290792, -6.832207, -7.626691, -2.731292, -5.258669], [-1.474883, -5.015319, -2.958467, -5.373497, -3.542351, -6.479951, -6.501667, -6.509994, -5.972708, -1.357894, -1.002569, -7.352375, -6.164752, -6.50269, -6.972933, -6.714185, -3.435756, -4.680408], [-2.035092, -4.338289, -3.211541, -2.200575, -4.949358, -5.656326, -5.396759, -5.249298, -4.176059, -1.967184, -0.865638, -5.176634, -4.248773, -4.006239, -6.457195, -4.241722, -3.210144, -4.237207], [-2.038199, -3.228688, -2.320211, -4.098757, -3.327213, -7.599248, -6.961544, -6.707355, -6.488754, -1.306977, -0.954661, -7.74556, -6.140603, -7.037845, -7.045454, -6.952082, -4.501697, -5.733513], [-2.225676, -6.514887, -2.464812, -6.170017, -3.426084, -6.798225, -6.712908, -7.430903, -7.301693, -1.449099, -0.754951, -6.999574, -5.223729, -6.739678, -7.090841, -6.169267, -3.300874, -4.185016], [-1.958164, -5.090039, -3.239037, -3.921324, -2.385787, -7.173878, -7.549573, -7.297557, -7.949902, -1.330592, -1.165884, -4.735598, -4.566328, -5.562993, -5.547321, -5.814373, -2.687893, -3.669676], [-1.415797, -5.124791, -3.72999, -5.937103, -2.77434, -7.611229, -6.684304, -7.228667, -7.021747, -1.041013, -1.274972, -6.768194, -6.002055, -6.97262, -6.71927, -7.166085, -4.092322, -5.757923], [-1.795759, -4.323911, -2.689996, -3.677934, -2.416073, -6.026309, -6.29922, -6.2174, -6.636759, -1.260487, -1.271233, -5.676086, -5.819088, -5.877975, -4.990658, -7.546271, -3.258427, -4.440696], [-2.61722, -4.011564, -3.06872, -7.382202, -2.033442, -9.04288, -7.250097, -6.208514, -6.490267, -1.58362, -0.795122, -7.336507, -5.409126, -7.75969, -7.979501, -7.938254, -2.878042, -4.921375], [-1.630303, -5.898397, -2.424395, -7.596372, -3.986821, -7.465317, -5.224844, -6.926117, -6.22892, -1.876089, -0.723826, -7.672456, -6.058282, -6.804053, -6.934775, -6.8057, -3.384853, -4.972162], [-3.520806, -5.821022, -5.236857, -3.905673, -5.203064, -6.460285, -5.477113, -5.407434, -4.804466, -1.441366, -0.800515, -5.962664, -5.387274, -5.26273, -6.688133, -6.136476, -1.966181, -2.568312], [-1.171876, -4.773542, -3.698785, -5.611339, -4.239188, -7.680113, -6.928211, -7.240894, -6.869536, -1.408171, -1.058222, -6.146851, -7.375213, -6.528236, -6.417329, -7.416067, -3.63036, -4.500265], [-1.886318, -5.586831, -1.98118, -6.628293, -4.517191, -8.585034, -8.018221, -8.239181, -7.67584, -1.779813, -0.826393, -6.58298, -3.938451, -5.027385, -5.790178, -5.216271, -3.125086, -4.947612], [-1.472729, -4.694489, -2.403356, -4.523839, -3.18975, -8.148925, -8.006869, -7.453607, -7.648386, -1.163111, -1.433154, -5.949156, -6.147856, -7.139911, -6.766676, -6.794388, -2.90207, -5.614103], [-1.738691, -4.707352, -3.826949, -4.135157, -4.883602, -5.57544, -6.960613, -6.988167, -7.407636, -1.901075, -0.742164, -3.247633, -5.531084, -5.457998, -5.262113, -5.723494, -2.771684, -3.911133], [-2.068664, -3.393206, -3.099137, -4.001811, -3.008611, -8.103941, -7.652669, -7.287173, -7.305216, -1.204047, -1.086465, -6.94589, -4.156649, -5.746489, -6.348117, -6.20814, -2.97739, -4.3207], [-1.399549, -4.651613, -3.312766, -4.702465, -3.989945, -6.866651, -6.531891, -6.188667, -6.659155, -1.698736, -1.029138, -4.36986, -4.584521, -5.500882, -5.641186, -6.328475, -2.423507, -4.357863], [-1.989339, -5.40788, -3.355859, -2.865795, -5.228551, -6.695207, -7.361637, -6.855638, -6.545956, -1.495775, -0.716905, -6.440009, -6.005673, -6.586775, -6.289982, -7.270035, -3.427495, -5.426523], [-1.852533, -4.816337, -2.12743, -4.117258, -3.676938, -6.433059, -6.721705, -6.58857, -6.269467, -1.699473, -0.860392, -5.167854, -5.348775, -6.136247, -6.063732, -6.270781, -3.401164, -4.397379], [-2.010576, -4.109333, -4.163293, -4.737053, -3.646829, -6.331767, -5.557328, -6.30626, -6.082826, -0.904745, -1.177802, -6.0855, -4.42242, -5.901696, -6.785455, -5.454913, -3.183101, -4.338737], [-1.857381, -4.222777, -2.317986, -4.401091, -4.15388, -7.274362, -6.474504, -6.025102, -6.75192, -1.230922, -1.457009, -5.870017, -4.361774, -6.283868, -5.428062, -5.747461, -2.044009, -4.044985], [-3.284824, -6.126933, -3.845961, -1.745259, -3.5562, -7.949265, -7.218443, -6.239327, -7.342067, -2.013862, -0.622502, -8.623096, -5.1184, -6.346183, -6.43399, -7.200349, -2.992304, -6.402938], [-2.023818, -4.914773, -3.413101, -3.259905, -4.358827, -6.054361, -7.417813, -6.42078, -6.579458, -1.297582, -1.132258, -4.042998, -3.75913, -4.113436, -4.36421, -3.448421, -2.711731, -4.992924], [-2.319717, -4.623295, -4.228964, -3.207441, -4.49996, -7.303796, -6.059953, -5.436936, -5.997886, -1.299476, -0.850435, -8.07046, -5.354358, -6.408445, -6.570272, -4.392385, -2.406382, -5.222841], [-2.386744, -6.260525, -5.929952, -2.186988, -4.475146, -8.093978, -6.757461, -7.535605, -6.789936, -1.125637, -1.168156, -5.193725, -5.701627, -6.470577, -7.33068, -6.554206, -2.169922, -4.223862], [-1.329894, -5.276568, -3.621767, -2.923773, -3.899077, -8.441214, -7.036023, -7.797641, -7.787804, -1.171332, -1.406447, -6.256862, -6.155261, -6.405545, -6.109757, -6.956773, -2.832729, -5.287577], [-1.254127, -4.649399, -3.13617, -4.511128, -3.601246, -6.579014, -7.00196, -6.972583, -7.002948, -1.281516, -1.281435, -5.884591, -5.295624, -6.169764, -6.238238, -5.555056, -3.085245, -5.969514], [-2.513538, -4.286253, -2.104501, -4.046094, -2.411659, -7.192772, -6.175496, -6.288893, -6.367118, -1.489084, -0.984032, -5.869786, -4.364015, -5.902218, -5.448457, -5.652471, -3.478135, -4.308613], [-1.934429, -5.378502, -3.868491, -5.091692, -3.124924, -7.260966, -6.700333, -7.009902, -7.066707, -1.324462, -0.847181, -6.138466, -6.694926, -7.551697, -7.269352, -7.30204, -2.701317, -4.684639], [-3.788042, -2.966868, -5.351829, -2.462923, -4.080179, -6.952316, -6.654277, -6.508501, -6.527453, -0.879196, -1.180093, -6.117682, -5.109572, -6.166391, -6.935572, -4.99582, -2.767261, -4.548055], [-2.612989, -2.977113, -2.514375, -2.460033, -2.813804, -5.50959, -5.525497, -5.319341, -4.464953, -1.01344, -1.657173, -4.902242, -5.195095, -5.373891, -6.096741, -6.070771, -3.261029, -4.525726], [-1.353636, -4.246641, -2.873059, -5.038308, -2.276368, -8.366526, -7.74009, -7.432296, -7.515623, -1.613389, -1.116812, -6.974594, -5.86804, -5.917324, -6.341733, -7.042575, -3.888543, -5.567971], [-1.744692, -3.962982, -3.694166, -2.985174, -5.557277, -5.524788, -6.716818, -5.890072, -6.850922, -1.292975, -1.131747, -5.162113, -4.597668, -4.881747, -5.067995, -4.891569, -2.523356, -5.597119], [-2.450485, -8.090636, -4.004327, -3.430823, -4.905591, -7.811103, -6.805826, -6.85489, -6.478775, -1.747005, -0.589067, -5.710822, -6.157324, -5.711887, -7.401062, -6.484418, -2.284186, -4.659771], [-1.829689, -3.36702, -2.404378, -4.713966, -2.909631, -6.500219, -6.547427, -5.871662, -6.038635, -1.119344, -1.440583, -6.608219, -4.950628, -5.780999, -6.256451, -5.356437, -3.029806, -4.315534], [-1.421635, -3.566341, -3.708603, -3.595527, -4.212451, -6.946845, -7.004967, -6.623947, -6.045486, -1.317756, -1.094963, -7.367984, -5.981893, -6.351293, -7.300043, -6.109774, -3.156118, -5.218886], [-2.306068, -2.818836, -2.958946, -4.962906, -2.388326, -8.198905, -7.257651, -6.044915, -6.467342, -1.408599, -1.029385, -5.139534, -5.176516, -6.485612, -5.791867, -6.647816, -2.804627, -5.203671], [-1.280763, -5.672918, -4.066663, -5.921844, -3.794648, -7.730683, -6.638751, -6.526997, -7.625809, -1.118026, -1.230341, -7.433519, -5.699864, -6.754696, -7.30774, -6.237002, -3.133954, -6.057315], [-2.279822, -4.699874, -3.339607, -3.165291, -3.250944, -6.802671, -5.412929, -5.239227, -5.649177, -1.216067, -1.228044, -5.511526, -5.705805, -5.428841, -5.409733, -6.1912, -1.979614, -4.409551], [-2.375836, -5.198111, -3.209789, -3.735093, -5.480172, -9.296831, -9.376444, -6.915088, -8.141886, -1.436945, -0.783883, -6.884992, -4.382167, -5.377007, -5.779805, -6.554206, -2.231897, -4.905182], [-1.397917, -3.57467, -2.887529, -3.896349, -4.80569, -5.049542, -5.435191, -6.485529, -5.376128, -1.439128, -1.216767, -5.483622, -5.211295, -5.484459, -5.717907, -5.347655, -3.084032, -3.779794], [-2.20699, -3.285312, -2.734468, -3.63832, -3.615567, -8.390056, -7.811776, -6.617891, -6.572706, -1.516788, -0.819687, -6.981956, -6.665879, -6.447387, -6.978088, -7.758696, -2.962662, -4.252059], [-1.967702, -3.841927, -3.946638, -4.107061, -4.119207, -7.995043, -7.105811, -6.264326, -6.414907, -1.362843, -1.146089, -6.14369, -5.562385, -5.858489, -7.066291, -6.582073, -1.745654, -3.788992], [-2.174322, -4.976705, -3.268154, -4.742935, -2.895964, -5.183914, -6.350126, -6.663727, -6.81952, -1.212315, -0.902345, -5.197173, -4.364545, -6.052668, -5.771894, -5.735037, -3.528821, -4.79777], [-2.020519, -5.069611, -3.034026, -5.359257, -4.458106, -6.627756, -6.312597, -6.478011, -6.783632, -1.228519, -0.932824, -6.449213, -5.220067, -5.563956, -6.914743, -6.341338, -2.525243, -4.498119], [-2.626472, -5.536901, -4.204812, -4.172324, -5.084553, -6.856585, -7.702682, -6.704056, -7.743407, -0.99059, -0.854867, -6.344332, -4.333928, -6.26803, -6.971271, -5.691449, -2.890651, -4.536236], [-2.42352, -3.194603, -4.64199, -2.184062, -4.882741, -4.260638, -5.350142, -5.487894, -6.366902, -1.280027, -1.320332, -4.654524, -3.66705, -4.024759, -3.001656, -4.986656, -3.021978, -4.370657], [-1.738239, -4.136361, -2.557298, -3.072738, -2.825865, -7.911953, -7.896591, -7.230063, -7.502712, -1.487503, -1.195585, -5.848879, -6.680928, -6.859087, -7.350323, -8.029033, -2.462565, -5.721616], [-2.451025, -6.196083, -2.835564, -3.766274, -2.699431, -6.776042, -5.535416, -7.114117, -5.642481, -1.388683, -0.813879, -7.009161, -6.665416, -6.991553, -6.828498, -7.210248, -3.096554, -4.548377], [-2.148868, -4.398582, -2.727831, -4.647542, -2.778448, -6.450178, -7.200463, -6.529144, -6.659929, -1.206033, -1.069281, -5.602067, -5.59215, -5.722198, -6.499753, -4.515626, -3.063244, -4.108257], [-2.28423, -5.981586, -3.252362, -5.257101, -4.21903, -9.579243, -8.628384, -7.407903, -8.317078, -1.209086, -0.733758, -7.915043, -6.56041, -7.963518, -8.219939, -8.374556, -3.025048, -5.089253], [-2.388629, -4.300745, -3.767717, -2.017101, -3.770525, -7.065702, -6.601226, -6.003426, -6.400801, -1.245961, -1.119384, -4.569149, -4.864123, -4.234704, -5.563516, -5.680229, -2.996164, -5.255912], [-2.980054, -2.602487, -4.345865, -4.873303, -1.789021, -5.61988, -5.947045, -5.365783, -5.959329, -1.402967, -1.268293, -6.489938, -4.41356, -5.944318, -5.886692, -3.500158, -2.745122, -3.403509], [-1.730542, -5.867208, -2.887937, -3.266638, -3.956689, -7.240444, -6.91826, -7.401803, -6.824926, -1.412216, -0.955873, -7.658394, -5.392924, -5.917053, -7.360471, -5.771262, -2.894459, -4.738425], [-1.203332, -5.043363, -4.884899, -2.729526, -5.570542, -7.688513, -7.055639, -7.159162, -7.498185, -1.438858, -1.176225, -7.266487, -6.316816, -6.015682, -6.797463, -6.620261, -2.809519, -7.109442], [-2.301884, -3.747929, -3.620326, -4.800616, -3.060941, -7.290874, -5.711325, -5.656929, -6.215411, -1.231861, -0.916975, -6.683506, -4.955763, -5.497444, -5.731092, -4.838609, -2.784046, -4.800793], [-2.069618, -5.804312, -3.896451, -5.176422, -4.64792, -7.892005, -7.707936, -7.611467, -7.552674, -1.009487, -0.905818, -7.33731, -6.095885, -7.123255, -7.304781, -7.338816, -2.878163, -5.66086], [-1.451906, -6.738896, -4.515529, -5.656286, -5.036609, -7.972946, -7.836291, -7.055192, -7.449028, -1.734966, -0.815684, -6.208245, -6.767125, -6.551493, -7.576547, -7.661194, -2.177073, -5.549791], [-1.556667, -4.565017, -2.3792, -7.409369, -3.628885, -6.324732, -5.39944, -7.166514, -6.048913, -1.22409, -1.141661, -6.752903, -5.986195, -6.934242, -7.277, -6.847128, -3.815651, -4.855078], [-1.843005, -5.329603, -5.499409, -2.679352, -5.184655, -6.763179, -6.919261, -5.968842, -6.142407, -1.707067, -0.820549, -4.957895, -5.888563, -5.473252, -6.253126, -7.143978, -2.257439, -4.738916], [-2.021849, -3.612123, -3.001591, -7.56335, -3.002852, -9.756319, -8.212197, -6.997505, -7.196961, -1.098092, -1.142551, -6.065798, -3.948852, -5.608173, -5.849725, -6.615116, -3.004038, -4.941042], [-1.503394, -4.43444, -3.218332, -3.088003, -2.746087, -8.656983, -7.789407, -7.183774, -7.305365, -1.237594, -1.360248, -6.574338, -5.625659, -6.24653, -6.587671, -6.444295, -2.968211, -5.120209], [-2.008333, -2.576955, -4.031131, -3.851289, -2.974449, -6.045543, -5.468884, -6.738117, -6.599686, -1.242478, -1.399766, -4.833203, -4.797389, -4.962404, -4.839277, -4.661637, -2.383874, -3.794994], [-1.947712, -5.028564, -4.455115, -2.543869, -4.905641, -6.076602, -6.511011, -6.907721, -7.214148, -1.103017, -1.189032, -5.314116, -4.196745, -4.452693, -5.567441, -4.212901, -2.829689, -6.168289], [-1.517682, -3.715363, -4.486637, -2.96874, -5.878045, -7.992125, -6.714599, -6.07773, -6.405289, -1.097825, -1.35486, -5.433833, -4.799575, -4.946274, -5.231938, -5.975264, -2.911899, -4.42578], [-1.627871, -2.807732, -4.006598, -4.330635, -3.066955, -7.019332, -6.298242, -6.185415, -6.143547, -1.187877, -1.43776, -5.235962, -4.866896, -5.387975, -5.512215, -5.934507, -2.534065, -4.380697], [-2.447639, -5.493801, -4.867921, -2.80531, -3.18475, -7.801044, -7.46162, -6.612675, -6.519777, -0.971386, -1.050085, -9.079286, -6.712562, -6.462759, -7.705197, -7.713051, -2.779898, -6.362614], [-2.279951, -3.963467, -2.953716, -2.594893, -3.83839, -6.63549, -6.255242, -5.171794, -6.109307, -1.175506, -1.114172, -6.469709, -4.987119, -6.239183, -6.505753, -6.318399, -2.737978, -5.501254], [-2.262686, -4.450401, -4.82808, -5.410225, -4.273133, -6.776459, -6.016412, -6.927522, -6.554716, -1.186659, -0.888676, -6.141605, -6.300451, -6.756502, -7.468073, -7.335968, -2.135457, -4.522675], [-1.262314, -4.781607, -3.195226, -3.879058, -5.050833, -6.773619, -7.424896, -7.147532, -8.036523, -1.128606, -1.368407, -6.910299, -6.003593, -6.455926, -6.658851, -6.45499, -2.996642, -6.269557], [-2.481075, -6.369296, -4.011749, -1.637472, -4.190559, -6.645534, -6.69869, -6.666147, -6.520715, -1.219115, -1.193057, -5.622369, -5.285729, -5.987984, -6.257684, -4.936202, -2.865838, -5.177988], [-1.635109, -2.52, -4.029047, -3.975745, -2.563386, -7.263317, -8.024461, -6.90001, -7.545973, -1.093252, -1.551536, -5.912023, -6.093674, -6.91634, -7.07604, -6.814695, -3.06454, -4.987224], [-1.599112, -3.21712, -3.596084, -3.110522, -3.726112, -5.991214, -6.489976, -5.846125, -5.736358, -1.092566, -1.662282, -5.567964, -4.326034, -5.017437, -5.18604, -5.471801, -2.519867, -4.371359], [-1.586653, -4.404129, -3.152005, -4.922125, -4.730074, -6.659067, -5.984138, -5.242791, -6.132111, -1.342981, -1.040765, -5.035663, -4.431146, -4.737479, -5.433919, -5.237215, -3.162424, -3.931631], [-2.66439, -5.131395, -2.223413, -3.413533, -2.8297, -7.151431, -6.809051, -5.902242, -6.123229, -1.300376, -1.059146, -6.949636, -5.04431, -6.16414, -6.884755, -4.739229, -2.569559, -6.025697], [-1.977367, -6.220567, -4.635125, -2.852424, -3.148965, -6.971561, -6.615949, -6.649365, -6.938329, -1.223563, -1.039261, -6.24568, -5.252626, -6.125421, -6.017912, -5.641465, -2.576717, -5.208429], [-2.194936, -3.876224, -3.589973, -2.143112, -2.235979, -7.10903, -6.369218, -5.611314, -6.154052, -1.12794, -1.507414, -7.247254, -6.267395, -5.909081, -6.180915, -4.511752, -3.218714, -5.459049], [-2.472792, -5.570397, -4.400457, -4.002402, -3.694136, -7.07937, -6.938354, -6.957435, -7.381895, -0.881661, -0.967148, -7.194767, -6.62734, -7.123338, -7.355188, -7.275273, -2.96116, -5.873612], [-1.2448, -4.210734, -4.54703, -4.204764, -5.316452, -5.363547, -5.384914, -6.527575, -6.609423, -1.146506, -1.526531, -5.317049, -5.10682, -5.183162, -4.849321, -5.093428, -2.47914, -5.247365], [-1.202147, -3.701302, -2.774863, -4.328816, -3.971214, -7.523936, -7.586317, -6.42226, -6.33184, -1.469608, -1.212664, -6.161952, -5.312385, -5.755885, -6.13231, -6.119668, -3.546691, -5.27747], [-1.580724, -6.217608, -3.394638, -3.587222, -5.457327, -8.0413, -8.415947, -7.208527, -7.498739, -1.316426, -0.967832, -7.458093, -4.194635, -5.952305, -6.388473, -7.083686, -2.953357, -5.542662], [-3.407973, -5.203153, -4.921859, -4.334644, -5.691187, -6.87091, -6.738201, -7.092227, -6.361596, -0.777926, -0.905844, -6.090644, -5.109926, -6.417105, -6.841842, -5.317038, -3.489984, -3.777691], [-1.98029, -4.247366, -5.199262, -3.741236, -5.141946, -7.846896, -7.81348, -7.418271, -7.511614, -1.101984, -1.073863, -5.575229, -3.369109, -4.96451, -4.934287, -4.308546, -2.708819, -5.453505], [-2.957603, -3.802062, -3.831817, -2.728447, -4.843588, -7.360528, -6.97969, -7.642459, -7.417561, -0.561568, -1.651181, -6.862503, -7.040592, -8.474045, -8.159967, -6.876238, -2.872465, -5.104322], [-1.66282, -5.439265, -2.495379, -2.979428, -2.450735, -6.006111, -6.689226, -6.781015, -6.168637, -1.158218, -1.519849, -6.782194, -5.930216, -6.405689, -7.267654, -6.563016, -3.333921, -5.625971], [-1.475903, -4.660101, -4.302697, -3.019859, -4.120406, -7.273403, -6.459996, -6.16267, -6.276729, -1.1261, -1.495212, -4.672563, -5.092301, -5.156065, -5.573891, -5.75264, -2.354162, -5.213346], [-3.103319, -2.120203, -4.32154, -5.333233, -4.273014, -7.996963, -6.096473, -5.264884, -5.91124, -1.449006, -1.062102, -6.096025, -4.439092, -4.917244, -6.173924, -5.688791, -2.313221, -2.450379], [-1.810304, -6.713021, -4.335063, -3.132987, -4.712895, -7.37791, -7.870233, -7.123676, -7.151549, -1.214676, -1.020075, -4.908769, -4.973301, -5.016283, -5.738477, -5.553453, -2.534825, -6.099951], [-2.500776, -6.765626, -4.676377, -2.770742, -3.509344, -7.415057, -6.990673, -7.872726, -5.601446, -1.399196, -0.802372, -6.238644, -5.404382, -5.987601, -6.91109, -5.466587, -2.456529, -4.239673], [-2.245932, -5.186615, -5.1661, -3.619727, -2.947766, -9.098685, -7.610029, -7.465533, -7.58233, -1.502997, -1.132822, -6.350031, -6.126873, -7.09672, -6.095959, -8.171751, -1.429819, -4.53682], [-2.299366, -5.082235, -2.387877, -3.106733, -4.813318, -6.770282, -7.347914, -6.011798, -6.169922, -1.505608, -1.144889, -5.548532, -5.570343, -5.477131, -5.816308, -5.294349, -1.748858, -4.773497], [-2.700227, -7.279748, -3.652657, -5.915604, -5.752295, -8.186601, -7.461915, -7.611236, -7.081929, -1.049996, -0.680932, -7.541843, -5.844124, -6.903884, -7.957234, -6.964163, -3.443384, -5.43288], [-1.968022, -4.84993, -1.991312, -5.865061, -3.104921, -7.889984, -6.020294, -5.921857, -6.159165, -1.385041, -1.045478, -6.563669, -5.607283, -6.459893, -6.503626, -7.434183, -3.096293, -5.324648], [-1.608689, -3.957411, -2.830553, -3.900939, -4.753267, -6.54683, -6.298302, -6.111008, -6.740471, -1.431707, -1.155215, -5.824619, -4.439992, -4.993049, -5.325979, -6.025043, -2.4312, -4.160566], [-2.469941, -4.559742, -2.8295, -5.12412, -2.51417, -5.669815, -5.444362, -6.049031, -5.20399, -1.144298, -1.055091, -7.800565, -6.135232, -7.058974, -6.99612, -5.458557, -2.937918, -4.186199], [-1.363707, -3.956221, -2.495936, -4.035044, -4.222103, -6.536203, -7.369971, -6.524441, -7.307133, -1.511322, -1.148839, -4.200945, -6.090064, -6.034406, -6.488441, -7.830668, -3.254297, -4.782662], [-1.733793, -3.375644, -2.866043, -1.929772, -3.72698, -7.733923, -8.750191, -6.864568, -6.809954, -1.134698, -1.563037, -7.706847, -6.6295, -6.913356, -7.056184, -7.362512, -3.832193, -5.703135], [-2.677525, -6.569466, -4.010957, -3.232195, -4.075322, -8.323105, -8.263736, -7.516071, -7.622732, -1.205193, -0.904261, -6.775967, -5.920247, -6.589279, -6.7522, -6.50894, -1.992585, -5.282324], [-2.197001, -5.905124, -4.305977, -2.609545, -3.895574, -5.867596, -6.764982, -6.069063, -6.782251, -1.242479, -0.945238, -6.424963, -4.520607, -5.55632, -4.825717, -5.372815, -2.860188, -4.856155], [-2.900938, -6.336995, -5.059988, -5.354517, -3.888471, -7.856899, -6.750351, -6.973205, -6.812499, -0.699378, -1.043323, -5.952099, -5.493773, -6.464308, -6.525974, -6.809901, -3.208143, -4.870562], [-2.613844, -4.109189, -4.35641, -4.172911, -3.552143, -6.971148, -5.959568, -6.589826, -6.645612, -0.722774, -1.348219, -6.314554, -4.228198, -6.013059, -6.093004, -5.98145, -2.740634, -4.269869], [-2.139654, -5.28523, -2.871187, -3.249154, -4.673761, -7.26732, -7.622051, -7.581153, -6.859291, -1.038926, -0.994419, -6.796216, -6.24484, -6.95476, -6.970672, -7.165049, -3.32538, -5.458673], [-2.392037, -5.520133, -4.878638, -2.065558, -5.354863, -7.28118, -6.443982, -6.834454, -6.285725, -0.959398, -1.158127, -6.237644, -7.331742, -6.295801, -7.013822, -7.595658, -3.009955, -4.839889], [-1.819068, -3.946838, -4.664417, -4.472268, -3.526489, -8.488801, -7.532219, -7.264528, -7.712995, -0.911193, -1.292029, -5.930552, -6.240622, -6.401008, -6.816215, -6.623053, -2.577305, -5.310626], [-2.795306, -5.502266, -4.252498, -5.453207, -5.607653, -6.081927, -5.641254, -6.045441, -6.172222, -1.719197, -0.863205, -4.955339, -4.260199, -4.124437, -4.258367, -4.80779, -2.281652, -1.96915], [-2.505751, -2.901114, -1.949251, -4.916032, -3.044678, -5.820934, -4.051405, -4.64473, -5.497081, -1.501594, -1.149032, -5.884648, -4.972262, -6.452272, -5.872599, -6.03007, -3.219038, -3.329505], [-1.763705, -5.300124, -3.170331, -3.478939, -3.144778, -7.502178, -7.120995, -7.292486, -6.632095, -1.150613, -1.126178, -5.880149, -5.842941, -6.312183, -6.343985, -6.569792, -3.056214, -5.134548], [-1.292912, -5.075495, -4.12582, -6.067929, -3.11778, -8.91427, -7.231856, -7.402071, -7.534654, -1.06412, -1.375189, -7.195394, -6.317647, -6.942615, -7.595098, -7.364724, -3.025822, -5.637714], [-1.887401, -3.670667, -4.816214, -5.725368, -4.108378, -6.459938, -5.345395, -5.133295, -5.355448, -1.334888, -1.022019, -5.949528, -5.728667, -5.439767, -5.495801, -5.160917, -2.101895, -4.342191], [-2.578137, -4.870067, -3.303714, -2.943641, -2.195604, -7.55336, -6.687575, -6.001984, -6.228758, -1.248542, -1.099771, -5.743921, -5.276976, -5.045688, -6.407533, -5.42713, -2.716016, -5.920388], [-2.630106, -3.788634, -4.041158, -2.535524, -4.651638, -7.258175, -7.194912, -7.066407, -7.677468, -0.80191, -1.298128, -4.569564, -5.064314, -6.76042, -5.694805, -6.841636, -3.287357, -4.196352], [-2.062596, -3.115009, -2.7365, -3.708468, -3.225126, -6.677617, -5.93976, -5.461955, -4.73766, -1.15973, -1.284429, -5.237997, -4.955364, -4.694655, -5.502422, -5.163404, -3.204551, -3.897927], [-3.02247, -5.900829, -4.523643, -4.067865, -6.856495, -6.955801, -7.193439, -7.255543, -7.014005, -1.157385, -0.63195, -7.29412, -5.41798, -6.502803, -6.306838, -5.611801, -3.158352, -4.147387]], "expected": ["glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3", "glm-5.3-flash", "glm-5.3", "glm-5.3", "glm-5.3", "glm-5.3", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "claude-fable-5", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3", "glm-5.3-flash", "glm-5.3", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3", "claude-fable-5", "glm-5.3-flash", "glm-5.3", "glm-5.3-flash", "glm-5.3", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3", "glm-5.3", "glm-5.3-flash", "claude-fable-5", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3", "glm-5.3", "glm-5.3-flash", "glm-5.3", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3", "glm-5.3", "glm-5.3", "glm-5.3-flash", "glm-5.3", "glm-5.3", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3", "glm-5.3-flash", "glm-5.3", "glm-5.3", "glm-5.3", "glm-5.3", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3", "glm-5.3-flash", "glm-5.3", "glm-5.3", "glm-5.3", "glm-5.3", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3", "claude-fable-5", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3", "glm-5.3-flash", "glm-5.3", "glm-5.3-flash", "glm-5.3", "glm-5.3", "glm-5.3", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3", "glm-5.3", "glm-5.3", "glm-5.3", "glm-5.3-flash", "glm-5.3", "glm-5.3-flash", "glm-5.3-flash", "glm-5.3", "glm-5.3-flash", "glm-5.3", "glm-5.3", "glm-5.3"]}')
work = Path('/kaggle/temp/check'); work.mkdir(parents=True, exist_ok=True)
train = pd.read_csv(TRAIN, dtype={'id': str}).set_index('id').loc[CHECK['ids']].reset_index()
train[['id', 'text']].to_csv(work / 'test.csv', index=False)
np.savez(work / 'final-logits.npz', ids=np.array(CHECK['ids']), labels=np.array(CHECK['labels']),
         logits=np.array(CHECK['logits']))
cmd = [sys.executable, str(V1_DIR / 'v1_stage.py'), '--test-csv', str(work / 'test.csv'),
       '--final-logits', str(work / 'final-logits.npz'), '--out', '/kaggle/working/check.parquet',
       '--report', '/kaggle/working/report.json', '--bin', str(V1_BIN), '--pipeline', str(V1_DIR / 'run_pipeline.py'),
       '--flash-segs', str(V1_FLASH / '*.gguf'), '--glm53-segs', str(V1_GLM53 / '*.gguf'),
       '--model', str(V1_DIR / 'll_model.json'), '--work', str(work / 'v1-work'), '--outdir', '/kaggle/working/v1',
       '--deadline-mono', repr(time.monotonic() + 4 * 3600), '--seq-batch', str(V1_SEQ_BATCH),
       '--ubatch-div', str(V1_UBATCH_DIV), '--max-cover', '1.0', '--min-texts', '10', '--max-changed', '1.0']
for _k, _v in V1_COST.items():
    cmd += ['--' + _k.replace('_', '-'), repr(_v)]
subprocess.run(cmd, env=V1_ENV, check=True)
rep = json.loads(Path('/kaggle/working/report.json').read_text())
print({k: v for k, v in rep.items() if not isinstance(v, dict)})
got = pd.read_parquet('/kaggle/working/check.parquet').set_index('id').loc[CHECK['ids'], 'model'].tolist()
agree = float(np.mean([a == b for a, b in zip(got, CHECK['expected'])]))
print(f"stage vs offline predictions: {agree:.3f} agree on {len(got)} rows; status {rep['status']}", flush=True)
assert rep['status'] == 'applied' and agree >= 0.90
